# House Prices - Advanced Regression Techniques


### importing libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

## 1. Data Retrieval
1. Data Retrieval

• Loading: Use pandas to read both train.csv and test.csv into separate DataFrames.

• Dataset Inspection: Print the dimensions (shape) of both datasets and display the first five
rows of the training set to verify the column structure.

• Memory Management: Identify column data types and demonstrate how to load the data
using specific dtype parameters to optimize memory usage (e.g., downcasting floats or using
category for categorical strings).

In [3]:
# Data Loading 
train_data= pd.read_csv("data/train.csv")
test_data= pd.read_csv("data/test.csv")


In [4]:
# shape of the dataset 
print("train data ", train_data.shape)
print("test data ", test_data.shape)


train data  (1460, 81)
test data  (1459, 80)


In [5]:
# show the first rows to understand the data 
train_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [6]:
# checking the data types and missing values
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [ ]:
# get memory usage of the dataset
mem_before = train_data.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage before optimization: {mem_before:.2f} MB")

#  deep = true ; to get the actual memory usage of the dataset, including the memory used by the objects in the dataset.

Memory usage before optimization: 3.43 MB


In [8]:
# analysis cols types 
num_cols = train_data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = train_data.select_dtypes(include=['object']).columns

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numerical columns: 38
Categorical columns: 43


In [ ]:
# down casting string columns to category type to reduce memory usage
for col in cat_cols:
    train_data[col] = train_data[col].astype('category')

In [ ]:
# Downcasting numerical columns to reduce memory usage
for col in num_cols:
    train_data[col] = pd.to_numeric(train_data[col], downcast='float')

In [12]:
mem_after = train_data.memory_usage(deep=True).sum() / 1024**2

print(f"Memory before: {mem_before:.2f} MB")
print(f"Memory after: {mem_after:.2f} MB")

Memory before: 3.43 MB
Memory after: 0.29 MB


### Memory Optimization

We optimized memory usage by:
- Downcasting numerical columns to more efficient types (e.g., float32 instead of float64)
- Converting categorical string columns into pandas 'category' dtype

This significantly reduces memory usage and improves computational efficiency.
Memory before: 3.43 MB
Memory after: 0.29 MB


In [19]:
# Drop ID
train_data.drop("Id", axis=1, inplace=True)
test_data.drop("Id", axis=1, inplace=True)

# Fix MSSubClass
train_data["MSSubClass"] = train_data["MSSubClass"].astype("category")
test_data["MSSubClass"] = test_data["MSSubClass"].astype("category")

KeyError: "['Id'] not found in axis"

In [20]:
# checking the data types and missing values after optimization
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   MSSubClass     1460 non-null   category
 1   MSZoning       1460 non-null   category
 2   LotFrontage    1201 non-null   float32 
 3   LotArea        1460 non-null   float32 
 4   Street         1460 non-null   category
 5   Alley          91 non-null     category
 6   LotShape       1460 non-null   category
 7   LandContour    1460 non-null   category
 8   Utilities      1460 non-null   category
 9   LotConfig      1460 non-null   category
 10  LandSlope      1460 non-null   category
 11  Neighborhood   1460 non-null   category
 12  Condition1     1460 non-null   category
 13  Condition2     1460 non-null   category
 14  BldgType       1460 non-null   category
 15  HouseStyle     1460 non-null   category
 16  OverallQual    1460 non-null   float32 
 17  OverallCond    1460 non-null   fl

# 2. Data Cleaning
* Missing Value Analysis: Identify columns with significant missing values. For features like
PoolQC, MiscFeature, Alley, and Fence, determine if "NaN" represents a missing value or a
specific category (e.g., "No Pool"). Apply appropriate fill strategies.

* Outlier Removal: Plot the distribution of the target variable SalePrice. Identify observations
that act as significant outliers (e.g., houses with extreme ground living area) and justify
whether they should be removed to improve model training.
* Data Transformation: The SalePrice distribution is right-skewed. Apply a logarithmic
transformation (np.log1p) to normalize the target variable.

 **Missing Value Analysis**: 
 
 Identify columns with significant missing values. 
 
 For features like : PoolQC, MiscFeature, Alley, and Fence,
 
  determine if "NaN" represents a missing value or a specific category (e.g., "No Pool"). 
  
  Apply appropriate fill strategies.

In [22]:
# check missing values
missing = train_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

In [ ]:
#  Calculate missing value percentages
missing_percent = (missing / len(train_data)) * 100
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

missing_df

,Missing Count,Missing %
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageType,81,5.547945
GarageYrBlt,81,5.547945
GarageFinish,81,5.547945


In [ ]:
# Columns with no feature value, and "none" is a valid value

cols_to_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
                'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                'BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1',
                'MasVnrType']

for col in cols_to_none:
    
    # Add "None" to the list of categories assigned to it in this column. 
    train_data[col] = train_data[col].cat.add_categories('None')
    
    # fill the missing values with 'none'
    train_data[col] = train_data[col].fillna('None')


In [ ]:
# Group by Neighborhood and fill missing LotFrontage with the median of that specific area
train_data["LotFrontage"] = train_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

# Fill missing numerical values with 0 where missing data indicates the feature doesn't exist
for col in ('GarageYrBlt', 'MasVnrArea'):
    train_data[col] = train_data[col].fillna(0)


C:\Users\pc home\AppData\Local\Temp\ipykernel_28252\3646984318.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_data["LotFrontage"] = train_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))


In [30]:
# col with little missing values , fill it with the most frequent values 
train_data['Electrical'] = train_data['Electrical'].fillna(train_data['Electrical'].mode()[0])


In [ ]:
#  is there still any missing values in the dataset ?
print(train_data.isnull().sum().max()) 




0
Series([], dtype: int64)


In [36]:
train_data.isnull().sum().sum()

np.int64(0)

**Do the same for the test data**

In [38]:
# نفس الكود للـ test_data
for col in cols_to_none:
    # test_data[col] = test_data[col].cat.add_categories('None')
    test_data[col] = test_data[col].fillna('None')

for col in ('GarageYrBlt', 'MasVnrArea'):
    test_data[col] = test_data[col].fillna(0)


# تعبئة التست بناءً على حسابات الترين (الأكثر احترافية)
train_medians = train_data.groupby("Neighborhood")["LotFrontage"].median()
test_data["LotFrontage"] = test_data.apply(
    lambda row: train_medians[row["Neighborhood"]] if pd.isnull(row["LotFrontage"]) else row["LotFrontage"], axis=1
)


C:\Users\pc home\AppData\Local\Temp\ipykernel_28252\3223295981.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_medians = train_data.groupby("Neighborhood")["LotFrontage"].median()


In [40]:
missing_test = test_data.isnull().sum()
print(missing_test[missing_test > 0])

MSZoning        4
Utilities       2
Exterior1st     1
Exterior2nd     1
BsmtFinSF1      1
BsmtFinSF2      1
BsmtUnfSF       1
TotalBsmtSF     1
BsmtFullBath    2
BsmtHalfBath    2
KitchenQual     1
Functional      2
GarageCars      1
GarageArea      1
SaleType        1
dtype: int64


In [39]:
test_data.isnull().sum().sum()

np.int64(22)

**Outlier Removal**: 
- Plot the distribution of the target variable SalePrice. 
- Identify observations that act as significant outliers (e.g., houses with extreme ground living area) and justify whether they should be removed to improve model training.